In [1]:
from pathlib import Path 
import pandas as pd 
import sys
import os

project_root = Path.cwd().parent.parent.parent
sys.path.insert(0, str(project_root))


In [2]:
base_path =  Path("../../../data/raw/wipo/ip_indicators").resolve()
staging_path = Path("../../../data/staging/wipo/ip_indicators").resolve()


base_path.mkdir(parents=True, exist_ok=True)
staging_path.mkdir(parents=True, exist_ok=True)

# Ensure pandas prints every single column
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print(base_path)
print(staging_path)


C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\raw\wipo\ip_indicators
C:\Users\shresthn\OneDrive - ISED-ISDE\Desktop\or_country_profiles-main\data\staging\wipo\ip_indicators


In [3]:
# Mapping for the short codes
type_map = {
    "industrial": "ID",
    "patent": "PA",
    "trademark": "TM"
}

In [4]:
# These specific indicators are Unilateral (Office only, no Origin)
unilateral_codes = ['patent_3', 'trademark_7', 'industrial_7']

In [5]:
files = [
    # --- Industrial Designs ---
    "industrial_1a- Direct applications_Count by filing office and applicant's origin_1980_2024.csv",       # Direct Applications
    "industrial_1b- Applications via the Hague system_Count by filing office and applicant's origin_1980_2024.csv", # Hague (Intl) Applications
    "industrial_2a- Registrations for direct applications_Count by filing office and applicant's origin_1980_2024.csv", # Direct Registrations
    "industrial_2b- Registrations for applications via the Hague system_Count by filing office and applicant's origin_1980_2024.csv", # Hague (Intl) Registrations
    "industrial_7 - Design registrations in force (by office)_Total count by filing office_1980_2024.csv",  # Total Designs in Force - Unilateral (Total by Office; Origin is World)

    # --- Patents ---
    "patent_1a- Direct applications_Count by filing office and applicant's origin_1980_2024.csv",          # Direct Applications
    "patent_1b- PCT national phase entries_Count by filing office and applicant's origin_1980_2024.csv",    # PCT (Intl) Applications
    "patent_2a- Grant for direct applications_Count by filing office and applicant's origin_1980_2024.csv", # Direct Grants
    "patent_2b- Grant for PCT national phase entries_Count by filing office and applicant's origin_1980_2024.csv", # PCT (Intl) Grants
    "patent_3 - Patents in force_Total count by filing office_1980_2024.csv",                               # Total Patents in Force - Unilateral (Total by Office; Origin is World)

    # --- Trademarks ---
    "trademark_1a- Direct applications_Count by filing office and applicant's origin_1980_2024.csv",       # Direct Applications
    "trademark_1b- Applications via the Madrid system_Count by filing office and applicant's origin_1980_2024.csv", # Madrid (Intl) Applications
    "trademark_2a- Registrations for direct applications_Count by filing office and applicant's origin_1980_2024.csv", # Direct Registrations
    "trademark_2b- Registrations for applications via the Madrid system_Count by filing office and applicant's origin_1980_2024.csv", # Madrid (Intl) Registrations
    "trademark_7 - Trademarks in force (by office)_Total count by filing office_1980_2024.csv"             # Trademarks in force - Unilateral (Total by Office; Origin is World)
]

In [7]:
for filename in files:
    path = base_path / filename
    if not path.exists():
        continue

    #  Read Data (Skip WIPO metadata rows)
    df = pd.read_csv(path, header=6)
    code_part = filename.split('-')[0].strip()

    #  Handle Unilateral Files (Special Alignment)
    if code_part in unilateral_codes:
        # If it's patent_3, drop the 4th column ('Type') to shift years left
        if code_part == 'patent_3' and len(df.columns) > 3:
            df.drop(df.columns[3], axis=1, inplace=True)
        
        # Rename the first three remaining columns
        df.columns.values[0:3] = ["Office", "office_code", "Origin"]
        df.drop("office_code", axis=1, inplace=True)
        df['Origin'] = 'World'
        
    else:
        # Bilateral Mapping
        df.columns.values[0:3] = ["Office", "office_code", "Origin"]
        df.drop("office_code", axis=1, inplace=True)

    #  Trim empty years from the right
    while df[df.columns[-1]].isna().all():
        df.drop(df.columns[-1], axis=1, inplace=True)

    #  Standardize Year Headers (Office, Origin, 1980, 1981...)
    num_year_cols = len(df.columns) - 2
    years = [str(1980 + i) for i in range(num_year_cols)]
    df.columns = ['Office', 'Origin'] + years

    #  Export to Parquet
    parts = code_part.split('_')
    if len(parts) >= 2:
        short_cat = type_map.get(parts[0], parts[0])
        new_name = f"stg_wipo__{short_cat}{parts[1]}.parquet"
        df.to_parquet(staging_path / new_name, index=False)
        print(f"Exported: {new_name}")

Exported: stg_wipo__ID1a.parquet
Exported: stg_wipo__ID1b.parquet
Exported: stg_wipo__ID2a.parquet
Exported: stg_wipo__ID2b.parquet
Exported: stg_wipo__ID7.parquet
Exported: stg_wipo__PA1a.parquet
Exported: stg_wipo__PA1b.parquet
Exported: stg_wipo__PA2a.parquet
Exported: stg_wipo__PA2b.parquet
Exported: stg_wipo__PA3.parquet
Exported: stg_wipo__TM1a.parquet
Exported: stg_wipo__TM1b.parquet
Exported: stg_wipo__TM2a.parquet
Exported: stg_wipo__TM2b.parquet
Exported: stg_wipo__TM7.parquet
